In [ ]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working directory:", os.getcwd())

In [ ]:
import sqlite3
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

DB_PATH = "results/variants.db"

def query(sql, params=()):
    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql_query(sql, conn, params=params)

print("Connected to:", DB_PATH)

## Cross-sample QC overview
Mapping rate, variant counts, and mean depth for every run in the database.

In [ ]:
runs = query("""
    SELECT r.run_id, s.name AS sample, r.mapping_rate,
           r.total_raw, r.total_filt,
           r.run_date, r.reference
    FROM runs r
    JOIN samples s ON r.sample_id = s.sample_id
    ORDER BY r.run_date DESC, r.run_id
""")

if runs.empty:
    print("No runs found in database.")
else:
    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=("Mapping rate (%)", "Filtered variants", "Raw vs filtered variants"),
        horizontal_spacing=0.10
    )

    # Mapping rate
    colors = ["#e63946" if v < 85 else "#2d6a4f" for v in runs["mapping_rate"]]
    fig.add_trace(go.Bar(
        x=runs["sample"], y=runs["mapping_rate"],
        marker_color=colors, name="Mapping rate",
        hovertemplate="%{x}<br>Mapping rate: %{y:.1f}%<extra></extra>"
    ), row=1, col=1)
    fig.add_hline(y=85, line_dash="dash", line_color="#e63946",
                  annotation_text="85% threshold", row=1, col=1)

    # Filtered variant counts
    fig.add_trace(go.Bar(
        x=runs["sample"], y=runs["total_filt"],
        marker_color="#457b9d", name="Filtered variants",
        hovertemplate="%{x}<br>Filtered variants: %{y:,}<extra></extra>"
    ), row=1, col=2)

    # Raw vs filtered stacked
    fig.add_trace(go.Bar(
        x=runs["sample"], y=runs["total_filt"],
        name="Kept", marker_color="#2d6a4f",
        hovertemplate="%{x}<br>Kept: %{y:,}<extra></extra>"
    ), row=1, col=3)
    fig.add_trace(go.Bar(
        x=runs["sample"], y=runs["total_raw"] - runs["total_filt"],
        name="Filtered out", marker_color="#e63946",
        hovertemplate="%{x}<br>Filtered: %{y:,}<extra></extra>"
    ), row=1, col=3)

    fig.update_layout(
        barmode="stack", height=420,
        title="Run QC overview",
        showlegend=False
    )
    fig.update_yaxes(range=[0, 105], row=1, col=1)
    fig.show()
    display(runs[["sample", "run_id", "mapping_rate", "total_raw", "total_filt", "run_date"]]
            .rename(columns={"mapping_rate": "mapping_%", "total_raw": "raw_vars", "total_filt": "filt_vars"})
            .style.format({"mapping_%": "{:.1f}", "raw_vars": "{:,}", "filt_vars": "{:,}"}))

## Per-run deep-dive
Select a run to explore depth distribution, allele frequencies, QUAL scores, and chromosomal variant distribution.

In [ ]:
run_ids = runs["run_id"].tolist() if not runs.empty else []

run_selector = widgets.Dropdown(
    options=run_ids,
    description="Run:",
    layout=widgets.Layout(width="350px")
)
out_perrun = widgets.Output()

def render_perrun(run_id):
    calls = query("""
        SELECT gc.depth, gc.quality, gc.allele_freq,
               v.variant_type, v.chromosome
        FROM genotype_calls gc
        JOIN variants v ON gc.variant_id = v.variant_id
        WHERE gc.run_id = ?
    """, (run_id,))

    if calls.empty:
        print(f"No variant calls found for {run_id}.")
        return

    main_chroms = [f"chr{i}" for i in list(range(1, 23)) + ["X", "Y", "M"]]
    calls_main = calls[calls["chromosome"].isin(main_chroms)]

    # SNP/INDEL counts
    type_counts = calls["variant_type"].value_counts().reset_index()
    type_counts.columns = ["type", "count"]

    fig = make_subplots(
        rows=2, cols=3,
        subplot_titles=(
            "Read depth distribution",
            "QUAL score distribution",
            "Variant types",
            "Allele frequency spectrum",
            "Variants per chromosome",
            "Depth vs QUAL"
        ),
        horizontal_spacing=0.10,
        vertical_spacing=0.15
    )

    # Depth histogram (cap at 99th percentile for readability)
    depth_cap = calls["depth"].quantile(0.99)
    fig.add_trace(go.Histogram(
        x=calls[calls["depth"] <= depth_cap]["depth"],
        nbinsx=50, marker_color="#457b9d", name="Depth"
    ), row=1, col=1)

    # QUAL distribution
    fig.add_trace(go.Histogram(
        x=calls["quality"], nbinsx=50,
        marker_color="#2d6a4f", name="QUAL"
    ), row=1, col=2)

    # Variant types pie
    fig.add_trace(go.Pie(
        labels=type_counts["type"],
        values=type_counts["count"],
        hole=0.4,
        marker_colors=["#2d6a4f", "#e63946", "#457b9d", "#f4a261"]
    ), row=1, col=3)

    # Allele frequency spectrum
    fig.add_trace(go.Histogram(
        x=calls["allele_freq"], nbinsx=40,
        marker_color="#f4a261", name="AF"
    ), row=2, col=1)

    # Variants per chromosome
    chrom_counts = (
        calls_main.groupby("chromosome").size()
        .reindex(main_chroms).fillna(0).reset_index()
    )
    chrom_counts.columns = ["chromosome", "count"]
    fig.add_trace(go.Bar(
        x=chrom_counts["chromosome"],
        y=chrom_counts["count"],
        marker_color="#6d6875", name="Variants"
    ), row=2, col=2)

    # Depth vs QUAL scatter (subsample for speed)
    sample_n = min(5000, len(calls))
    scatter_df = calls.sample(sample_n, random_state=42)
    fig.add_trace(go.Scatter(
        x=scatter_df["depth"],
        y=scatter_df["quality"],
        mode="markers",
        marker=dict(size=3, opacity=0.4, color="#1d3557"),
        name="calls"
    ), row=2, col=3)

    fig.update_layout(
        height=700,
        title=f"Per-run deep-dive: {run_id}  ({len(calls):,} variant calls)",
        showlegend=False
    )
    fig.update_xaxes(title_text="Depth", row=1, col=1)
    fig.update_xaxes(title_text="QUAL", row=1, col=2)
    fig.update_xaxes(title_text="Allele frequency", row=2, col=1)
    fig.update_xaxes(tickangle=45, row=2, col=2)
    fig.update_xaxes(title_text="Depth", row=2, col=3)
    fig.update_yaxes(title_text="QUAL", row=2, col=3)
    fig.show()

    # Stats summary
    stats = pd.DataFrame([{
        "total_calls": len(calls),
        "median_depth": round(calls["depth"].median(), 1),
        "mean_depth": round(calls["depth"].mean(), 1),
        "median_qual": round(calls["quality"].median(), 1),
        "pct_af_gt50": round((calls["allele_freq"] > 0.5).mean() * 100, 1)
    }])
    display(stats)

def on_run_change(change):
    with out_perrun:
        out_perrun.clear_output(wait=True)
        render_perrun(change["new"])

run_selector.observe(on_run_change, names="value")
display(run_selector, out_perrun)

if run_ids:
    with out_perrun:
        render_perrun(run_ids[0])

## Cross-sample comparison
Select two runs to compare their QC metrics side-by-side and see how many variants they share.

In [ ]:
sel_a = widgets.Dropdown(
    options=run_ids, description="Run A:",
    layout=widgets.Layout(width="350px")
)
sel_b = widgets.Dropdown(
    options=run_ids,
    value=run_ids[1] if len(run_ids) > 1 else run_ids[0],
    description="Run B:",
    layout=widgets.Layout(width="350px")
)
out_compare = widgets.Output()

def render_compare(run_a, run_b):
    def get_variants(rid):
        df = query("""
            SELECT v.chromosome, v.position, v.ref_allele, v.alt_allele,
                   gc.depth, gc.allele_freq, gc.quality
            FROM genotype_calls gc
            JOIN variants v ON gc.variant_id = v.variant_id
            WHERE gc.run_id = ?
        """, (rid,))
        return df

    df_a = get_variants(run_a)
    df_b = get_variants(run_b)

    if df_a.empty or df_b.empty:
        print("One or both runs have no variant data.")
        return

    key_cols = ["chromosome", "position", "ref_allele", "alt_allele"]
    set_a = set(df_a[key_cols].itertuples(index=False, name=None))
    set_b = set(df_b[key_cols].itertuples(index=False, name=None))
    shared = len(set_a & set_b)
    only_a = len(set_a - set_b)
    only_b = len(set_b - set_a)

    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=("Variant overlap", "Depth comparison", "AF comparison"),
        horizontal_spacing=0.12
    )

    # Stacked overlap bar
    labels = [f"{run_a} only", "Shared", f"{run_b} only"]
    values = [only_a, shared, only_b]
    fig.add_trace(go.Bar(
        x=labels, y=values,
        marker_color=["#457b9d", "#2d6a4f", "#e63946"],
        text=[f"{v:,}" for v in values],
        textposition="auto"
    ), row=1, col=1)

    # Depth box comparison
    depth_cap = max(df_a["depth"].quantile(0.99), df_b["depth"].quantile(0.99))
    fig.add_trace(go.Box(
        y=df_a[df_a["depth"] <= depth_cap]["depth"],
        name=run_a, marker_color="#457b9d", boxmean=True
    ), row=1, col=2)
    fig.add_trace(go.Box(
        y=df_b[df_b["depth"] <= depth_cap]["depth"],
        name=run_b, marker_color="#e63946", boxmean=True
    ), row=1, col=2)

    # Allele frequency overlay
    fig.add_trace(go.Histogram(
        x=df_a["allele_freq"], nbinsx=30,
        name=run_a, opacity=0.6, marker_color="#457b9d"
    ), row=1, col=3)
    fig.add_trace(go.Histogram(
        x=df_b["allele_freq"], nbinsx=30,
        name=run_b, opacity=0.6, marker_color="#e63946"
    ), row=1, col=3)

    overlap_pct = shared / len(set_a) * 100 if set_a else 0
    jaccard = shared / len(set_a | set_b) * 100 if (set_a | set_b) else 0

    fig.update_layout(
        height=420,
        barmode="overlay",
        title=f"{run_a}  vs  {run_b}  |  overlap {overlap_pct:.1f}%  |  Jaccard {jaccard:.1f}%"
    )
    fig.show()

    # QC table
    run_info = query("""
        SELECT r.run_id, s.name AS sample, r.mapping_rate,
               r.total_raw, r.total_filt
        FROM runs r JOIN samples s ON r.sample_id = s.sample_id
        WHERE r.run_id IN (?, ?)
    """, (run_a, run_b))
    display(run_info.style.format({"mapping_rate": "{:.1f}",
                                   "total_raw": "{:,}", "total_filt": "{:,}"}))

def on_compare_change(change):
    with out_compare:
        out_compare.clear_output(wait=True)
        render_compare(sel_a.value, sel_b.value)

sel_a.observe(on_compare_change, names="value")
sel_b.observe(on_compare_change, names="value")

display(widgets.HBox([sel_a, sel_b]), out_compare)

if len(run_ids) >= 2:
    with out_compare:
        render_compare(run_ids[0], run_ids[1])

## STAR alignment QC
Spliced read percentage and short-read fraction across all runs (from STAR log).

In [ ]:
star_qc = query("""
    SELECT r.run_id, s.name AS sample,
           r.mapping_rate
    FROM runs r
    JOIN samples s ON r.sample_id = s.sample_id
    ORDER BY r.run_date DESC, r.run_id
""")

# Try to pull splice/short metrics from STAR logs directly
import re

def parse_star_log(run_id, sample):
    log_path = f"results/{run_id}/star/{sample}/Log.final.out"
    if not os.path.exists(log_path):
        return None, None
    text = open(log_path).read()
    m_splice = re.search(r"% of reads mapped to multiple loci.*?([\d.]+)%", text)
    m_short  = re.search(r"% of reads unmapped: too short.*?([\d.]+)%", text)
    m_annot  = re.search(r"% of splices: Annotated \(sjdb\).*?([\d.]+)%", text)
    splice = float(m_annot.group(1)) if m_annot else None
    short  = float(m_short.group(1))  if m_short  else None
    return splice, short

star_qc = query("""
    SELECT r.run_id, s.name AS sample, r.mapping_rate
    FROM runs r JOIN samples s ON r.sample_id = s.sample_id
    ORDER BY r.run_id
""")

splice_vals, short_vals = [], []
for _, row in star_qc.iterrows():
    sp, sh = parse_star_log(row["run_id"], row["sample"])
    splice_vals.append(sp)
    short_vals.append(sh)

star_qc["annotated_splice_pct"] = splice_vals
star_qc["pct_too_short"] = short_vals

has_star_data = star_qc["annotated_splice_pct"].notna().any()

if has_star_data:
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Annotated splice % (higher = better)",
                        "% reads too short (lower = better)"),
        horizontal_spacing=0.12
    )
    fig.add_trace(go.Bar(
        x=star_qc["sample"],
        y=star_qc["annotated_splice_pct"],
        marker_color="#2d6a4f"
    ), row=1, col=1)
    fig.add_trace(go.Bar(
        x=star_qc["sample"],
        y=star_qc["pct_too_short"],
        marker_color="#e63946"
    ), row=1, col=2)
    fig.update_layout(height=380, showlegend=False, title="STAR alignment QC")
    fig.show()
else:
    print("STAR log files not found or metrics not yet parsed.")
    print("Run the pipeline first, then re-execute this cell.")

display(star_qc)

## Raw SQL explorer
Run any query against the database.

In [ ]:
sql_box = widgets.Textarea(
    value="SELECT s.name, r.run_id, r.mapping_rate, r.total_filt\nFROM runs r JOIN samples s ON r.sample_id = s.sample_id\nORDER BY r.run_date DESC;",
    layout=widgets.Layout(width="100%", height="100px")
)
run_btn = widgets.Button(description="Run query", button_style="primary")
out_sql = widgets.Output()

def on_run_query(btn):
    with out_sql:
        out_sql.clear_output(wait=True)
        try:
            result = query(sql_box.value)
            display(result)
        except Exception as e:
            print(f"Error: {e}")

run_btn.on_click(on_run_query)
display(sql_box, run_btn, out_sql)